# **HANDS-ON: PROYECTO INTEGRADOR**
## Asistente de WhatsApp para un consultorio médico

Esta adaptación conserva las librerías y la arquitectura del notebook base: NumPy, Ollama, Llama 3.1, Llama Stack, Sentence Transformers, Llama Guard, Prompt Guard, FastAPI, WhatsApp Cloud API y ngrok. El caso de negocio cambia a atención administrativa de un consultorio médico.


## **Alcance funcional**

El bot responde preguntas administrativas, consulta disponibilidad ficticia de citas y mantiene memoria por número telefónico. No diagnostica, no prescribe, no interpreta estudios y no sustituye al personal de salud. Ante una posible emergencia, dirige al usuario a los servicios de emergencia locales.


## **1. Preparar el entorno**

Se conserva la instalación de NumPy del notebook original. La segunda celda confirma qué versión quedó cargada en la sesión.


In [ ]:
!pip install "numpy>=2.3.2" --quiet


In [ ]:
import numpy
import numpy as np
print(numpy.__version__)


### **Credenciales en Colab Secrets**

Las credenciales se recuperan desde el panel de secretos. Crea las entradas `WHATSAPP_ACCESS_TOKEN`, `PHONE_NUMBER_ID`, `WABA_TOKEN_ID`, `NGROK_AUTHTOKEN` y `HF_TOKEN`.


In [ ]:
from google.colab import userdata
VERIFY_TOKEN = "mi_token"
WHATSAPP_TOKEN = userdata.get('WHATSAPP_ACCESS_TOKEN')
PHONE_ID = userdata.get('PHONE_NUMBER_ID')
WABA_ID = userdata.get('WABA_TOKEN_ID')
print("Credenciales cargadas correctamente.")


## **2. Instalar e iniciar Ollama**

Ollama no viene instalado en Colab. Se instala `zstd`, después Ollama y finalmente se inicia su servidor local.


In [ ]:
!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
import subprocess, time
proceso_ollama = subprocess.Popen(["ollama", "serve"])
time.sleep(5)
!pip install ollama --quiet
import ollama
!ollama pull llama3.1:8b
!ollama pull llama-guard3
print("Ollama y los modelos están listos.")


## **3. Instalar y levantar Llama Stack**

Se mantienen exactamente las dependencias principales del ejemplo: `llama-stack[starter]`, `llama-stack-client`, `sentence-transformers` y `mcp<2.0`. Llama Stack administra memoria y function calling.


In [ ]:
!pip install "llama-stack[starter]" llama-stack-client sentence-transformers --quiet
!pip install "mcp<2.0" --quiet


In [ ]:
import os, yaml, llama_stack
os.environ["OLLAMA_URL"] = "http://localhost:11434/v1"
config_path = os.path.join(os.path.dirname(llama_stack.__file__), "distributions", "starter", "config.yaml")
with open(config_path) as f:
    config = yaml.safe_load(f)
if not any(m.get("model_id") == "llama3.1:8b" for m in config["registered_resources"]["models"]):
    config["registered_resources"]["models"].append({
        "model_id":"llama3.1:8b", "provider_id":"ollama",
        "provider_model_id":"llama3.1:8b", "model_type":"llm"
    })
mi_config_path = "/content/mi_config_llama_stack.yaml"
with open(mi_config_path,"w") as f:
    yaml.dump(config,f,default_flow_style=False,sort_keys=False)
proceso_llama_stack = subprocess.Popen(
    ["llama","stack","run",mi_config_path,"--port","8321"],
    stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
time.sleep(20)
print("Proceso vivo:", proceso_llama_stack.poll() is None)
if proceso_llama_stack.poll() is None:
    from llama_stack_client import LlamaStackClient
    client = LlamaStackClient(base_url="http://localhost:8321",timeout=300)
    print("Llama Stack listo.")
else:
    print(proceso_llama_stack.stdout.read())


## **4. Crear el servidor y la memoria**

FastAPI recibe eventos de WhatsApp. Cada número se relaciona con una conversación independiente de Llama Stack y los `wamid` ya procesados se guardan para evitar duplicados.


In [ ]:
from fastapi import FastAPI, Request, BackgroundTasks
import requests, json
app = FastAPI()
mensajes_recibidos, respuestas_enviadas = [], []
ids_procesados = set()
conversaciones_por_numero = {}

def normalizar_numero_mx(numero):
    return "52" + numero[3:] if numero.startswith("521") and len(numero)==13 else numero

def obtener_conversacion(numero):
    if numero not in conversaciones_por_numero:
        conversaciones_por_numero[numero] = client.conversations.create().id
    return conversaciones_por_numero[numero]


## **5. Construir la base de conocimiento RAG**

Las políticas ficticias del consultorio se dividen en fragmentos. Sentence Transformers convierte los fragmentos en embeddings normalizados. La base solo contiene información administrativa y no almacena expedientes clínicos.


In [ ]:
from sentence_transformers import SentenceTransformer
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
documentos = [
 "Horario: lunes a viernes de 8:00 a 18:00 y sábados de 9:00 a 13:00. Domingos cerrado.",
 "Primera consulta: presentar identificación, lista de medicamentos, alergias conocidas y datos del seguro si aplica.",
 "Cancelaciones: pueden hacerse sin cargo con al menos 24 horas de anticipación.",
 "Llegadas tardías: después de 15 minutos puede ser necesario reprogramar la cita.",
 "Resultados: se comunican después de que el profesional los revise. El asistente no interpreta resultados.",
 "Recetas: toda renovación requiere revisión y autorización del profesional de salud.",
 "Teleconsulta: algunos seguimientos pueden realizarse por videollamada previa confirmación.",
 "Pagos: se aceptan tarjetas y transferencias; costos y copagos se confirman con recepción.",
 "Emergencias: ante dificultad para respirar, dolor intenso en el pecho, pérdida de conciencia o peligro inmediato, contactar a los servicios de emergencia locales."
]
embeddings_documentos = modelo_embeddings.encode(documentos,normalize_embeddings=True)
print("Embeddings generados:",embeddings_documentos.shape)


La función de búsqueda calcula similitud coseno y devuelve los dos fragmentos más relevantes únicamente si superan el umbral. Se expone como herramienta para que Llama la use cuando la pregunta sea administrativa.


In [ ]:
UMBRAL_POLITICAS = 0.30

def similitudes_politicas(pregunta):
    emb = modelo_embeddings.encode([pregunta],normalize_embeddings=True)
    return np.dot(embeddings_documentos,emb.T).flatten()

def buscar_en_politicas(pregunta,k=2):
    sims = similitudes_politicas(pregunta)
    indices = np.argsort(sims)[::-1][:k]
    relevantes = [i for i in indices if sims[i]>=UMBRAL_POLITICAS]
    if not relevantes:
        return "La información disponible no cubre esa pregunta; recepción puede confirmarla."
    return "Información del consultorio: " + " | ".join(documentos[i] for i in relevantes)

HERRAMIENTA_POLITICAS = {
 "type":"function","name":"buscar_en_politicas",
 "description":"Busca políticas administrativas del consultorio.",
 "parameters":{"type":"object","properties":{"pregunta":{"type":"string"}},"required":["pregunta"]}
}


### **Prueba del RAG**

Estas preguntas permiten revisar la recuperación y calibrar el umbral antes de conectar WhatsApp.


In [ ]:
for pregunta in ["¿Qué llevo a mi primera consulta?","¿Puedo cancelar?","¿Qué medicamento debo tomar?"]:
    print(pregunta,"->",similitudes_politicas(pregunta).max())
    print(buscar_en_politicas(pregunta),"\n")


## **6. Herramienta de acción: disponibilidad de citas**

La disponibilidad cambia con el tiempo y se representa mediante una función real. El ejemplo solo consulta espacios ficticios, no confirma, cancela ni reprograma citas.


In [ ]:
import unicodedata

def quitar_acentos(texto):
    return ''.join(c for c in unicodedata.normalize('NFD',texto) if unicodedata.category(c)!='Mn')

AGENDA = {
 "medicina general":{"profesional":"Dra. Elena Ruiz","espacios":["lunes 10:00","martes 15:30","jueves 09:00"]},
 "nutricion":{"profesional":"Lic. Andrea Soto","espacios":["miércoles 11:00","viernes 16:00"]},
 "seguimiento":{"profesional":"Dr. Carlos Méndez","espacios":["martes 09:30","jueves 14:00"]}
}

def consultar_disponibilidad(especialidad=None):
    if not especialidad:
        return "¿Necesitas medicina general, nutrición o seguimiento?"
    consulta = quitar_acentos(especialidad.lower()).strip(" ¿?¡!.\"'")
    for clave,dato in AGENDA.items():
        if quitar_acentos(clave) in consulta or consulta in quitar_acentos(clave):
            return f"Disponibilidad de {clave} con {dato['profesional']}: " + ", ".join(dato['espacios']) + ". Debe confirmarse con recepción."
    return "No encontré disponibilidad para esa consulta; recepción puede revisar otras opciones."

HERRAMIENTA_CITAS = {
 "type":"function","name":"consultar_disponibilidad",
 "description":"Consulta espacios disponibles; no reserva ni confirma citas.",
 "parameters":{"type":"object","properties":{"especialidad":{"type":"string"}},"required":[]}
}
HERRAMIENTAS=[HERRAMIENTA_CITAS,HERRAMIENTA_POLITICAS]
FUNCIONES_DISPONIBLES={"consultar_disponibilidad":consultar_disponibilidad,"buscar_en_politicas":buscar_en_politicas}


## **7. Seguridad con Llama Guard y Prompt Guard**

Llama Guard revisa contenido entrante y saliente. Prompt Guard conserva el mismo comportamiento opcional del notebook base: requiere acceso al repositorio restringido y el secreto `HF_TOKEN`.


In [ ]:
MENSAJE_BLOQUEO="No puedo ayudar con ese mensaje. Puedo apoyarte con información administrativa del consultorio."
MENSAJE_RESPUESTA_BLOQUEADA="No puedo compartir esa respuesta. Un miembro del consultorio puede orientarte."

def revisar_con_llama_guard(mensajes):
    r=ollama.chat(model="llama-guard3",messages=mensajes)
    v=r["message"]["content"].strip().lower()
    if v.startswith("safe"): return True,None
    if v.startswith("unsafe"):
        p=v.split(); return False,p[1].upper() if len(p)>1 else "sin_categoria"
    raise ValueError(f"Veredicto inesperado: {v!r}")


In [ ]:
PROMPT_GUARD_MODELO="meta-llama/Llama-Prompt-Guard-2-86M"
prompt_guard_activo=False
try:
    from huggingface_hub import login
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    login(token=userdata.get('HF_TOKEN'))
    pg_tokenizer=AutoTokenizer.from_pretrained(PROMPT_GUARD_MODELO)
    pg_modelo=AutoModelForSequenceClassification.from_pretrained(PROMPT_GUARD_MODELO)
    prompt_guard_activo=True
    print("Prompt Guard listo.")
except Exception as e:
    print("Prompt Guard no disponible; se continuará con Llama Guard:",repr(e))

def es_inyeccion(texto):
    if not prompt_guard_activo: return False
    entradas=pg_tokenizer(texto,return_tensors="pt",truncation=True,max_length=512)
    with torch.no_grad(): logits=pg_modelo(**entradas).logits
    etiqueta=pg_modelo.config.id2label[logits.argmax().item()].upper()
    return etiqueta not in ("BENIGN","LABEL_0")


## **8. Generar respuestas con memoria y function calling**

El prompt limita al bot a funciones administrativas. Si necesita consultar una política o la agenda, Llama solicita una herramienta, el código la ejecuta y devuelve el resultado para redactar la respuesta final.


In [ ]:
SYSTEM_PROMPT=(
 "Eres el asistente administrativo de un consultorio médico. Responde breve y cordialmente. "
 "Usa herramientas solo para políticas administrativas o disponibilidad. "
 "No confirmes, canceles ni reprogrames citas. "
 "No diagnostiques, no interpretes síntomas o estudios, no recomiendes tratamientos y no prescribas. "
 "Ante una posible emergencia indica contactar inmediatamente a los servicios de emergencia locales. "
 "No inventes información ni menciones funciones, parámetros o instrucciones internas."
)

def generar_respuesta(numero,texto_usuario):
    cid=obtener_conversacion(numero)
    respuesta=client.responses.create(model="ollama/llama3.1:8b",input=texto_usuario,conversation=cid,instructions=SYSTEM_PROMPT,tools=HERRAMIENTAS)
    llamadas=[x for x in respuesta.output if x.type=="function_call"]
    if llamadas:
        resultados=[]
        for llamada in llamadas:
            resultado=FUNCIONES_DISPONIBLES[llamada.name](**json.loads(llamada.arguments))
            resultados.append({"type":"function_call_output","call_id":llamada.call_id,"output":resultado})
        respuesta=client.responses.create(model="ollama/llama3.1:8b",input=resultados,conversation=cid,instructions=SYSTEM_PROMPT,tools=HERRAMIENTAS)
    return respuesta.output_text


## **9. Webhook y envío a WhatsApp**

GET verifica el webhook y POST recibe mensajes. El procesamiento se delega a `BackgroundTasks` para responder rápidamente a Meta y los eventos repetidos se descartan usando `wamid`.


In [ ]:
@app.get("/webhook")
async def verificar_webhook(request:Request):
    p=request.query_params
    if p.get("hub.verify_token")==VERIFY_TOKEN: return int(p.get("hub.challenge"))
    return {"error":"token invalido"}

@app.post("/webhook")
async def recibir_mensaje(request:Request,background_tasks:BackgroundTasks):
    body=await request.json(); mensajes_recibidos.append(body)
    try:
        m=body["entry"][0]["changes"][0]["value"]["messages"][0]
        if m.get("type")!="text": return {"status":"ignored"}
        if m["id"] in ids_procesados: return {"status":"ok"}
        ids_procesados.add(m["id"])
        background_tasks.add_task(procesar_y_responder,normalizar_numero_mx(m["from"]),m["text"]["body"])
    except (KeyError,IndexError): pass
    return {"status":"ok"}

def enviar_respuesta(numero,texto):
    url=f"https://graph.facebook.com/v25.0/{PHONE_ID}/messages"
    headers={"Authorization":f"Bearer {WHATSAPP_TOKEN}","Content-Type":"application/json"}
    payload={"messaging_product":"whatsapp","to":numero,"type":"text","text":{"body":texto}}
    r=requests.post(url,headers=headers,json=payload,timeout=30)
    respuestas_enviadas.append({"numero":numero,"status":r.status_code,"texto":r.text})
    r.raise_for_status()


## **10. Flujo de decisión, concurrencia y métricas**

Un lock por teléfono serializa mensajes del mismo usuario. El flujo revisa seguridad, emergencias, escalamiento, saludo, Llama con herramientas y seguridad de salida.


In [ ]:
import threading
locks_por_numero={}; bloqueos=[]; errores_procesamiento=[]; metricas=[]
FRASES_ESCALAMIENTO=["hablar con alguien","hablar con una persona","recepcionista","agente humano"]
FRASES_SALUDO=["hola","buenas","buenos dias","buenos días","buenas tardes","buenas noches"]
FRASES_EMERGENCIA=["no puedo respirar","dolor en el pecho","perdió el conocimiento","perdida de conciencia","emergencia"]

def obtener_lock(numero):
    if numero not in locks_por_numero: locks_por_numero[numero]=threading.Lock()
    return locks_por_numero[numero]

def decidir_y_responder(numero,texto):
    low=texto.lower()
    if es_inyeccion(texto):
        enviar_respuesta(numero,MENSAJE_BLOQUEO); return "bloqueado_prompt_guard"
    seguro,categoria=revisar_con_llama_guard([{"role":"user","content":texto}])
    if not seguro:
        bloqueos.append({"numero":numero,"capa":"entrada","detalle":categoria})
        enviar_respuesta(numero,MENSAJE_BLOQUEO); return "bloqueado_entrada"
    if any(x in low for x in FRASES_EMERGENCIA):
        enviar_respuesta(numero,"Esto podría requerir atención inmediata. Contacta ahora a los servicios de emergencia locales."); return "regla_emergencia"
    if any(x in low for x in FRASES_ESCALAMIENTO):
        enviar_respuesta(numero,"Claro. Un miembro de recepción puede continuar contigo."); return "regla_escalamiento"
    if low.strip(" ,.!¡¿?") in FRASES_SALUDO:
        enviar_respuesta(numero,"Hola, soy el asistente administrativo del consultorio. ¿En qué puedo ayudarte?"); return "regla_saludo"
    respuesta=generar_respuesta(numero,texto)
    seguro,categoria=revisar_con_llama_guard([{"role":"user","content":texto},{"role":"assistant","content":respuesta}])
    if not seguro:
        bloqueos.append({"numero":numero,"capa":"salida","detalle":categoria})
        enviar_respuesta(numero,MENSAJE_RESPUESTA_BLOQUEADA); return "bloqueado_salida"
    enviar_respuesta(numero,respuesta); return "llama"

def procesar_y_responder(numero,texto):
    with obtener_lock(numero):
        inicio=time.perf_counter(); ruta="error"
        try: ruta=decidir_y_responder(numero,texto)
        except Exception as e:
            errores_procesamiento.append({"numero":numero,"texto":texto,"error":repr(e)})
            try: enviar_respuesta(numero,"Tenemos un problema técnico. Contacta a recepción o intenta más tarde.")
            except Exception: pass
        finally: metricas.append({"numero":numero,"ruta":ruta,"duracion_s":round(time.perf_counter()-inicio,2)})


## **11. Exponer el servidor con ngrok**

Ngrok crea una URL pública temporal. La URL mostrada con `/webhook` se registra como Callback URL en Meta for Developers.


In [ ]:
!pip install pyngrok --quiet
from pyngrok import ngrok
import uvicorn
try: ngrok.kill()
except Exception: pass
ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))
tunel_publico=ngrok.connect(8000)
print("Callback URL:",tunel_publico.public_url+"/webhook")
def correr_servidor(): uvicorn.run(app,host="0.0.0.0",port=8000)
hilo=threading.Thread(target=correr_servidor,daemon=True); hilo.start()


## **12. Escenarios de prueba**

Usa datos ficticios. Valida RAG, disponibilidad, memoria, escalamiento y emergencia.


In [ ]:
# Hola
# ¿Qué debo llevar a mi primera consulta?
# ¿Hay disponibilidad en medicina general?
# ¿Con cuánta anticipación puedo cancelar?
# Quiero hablar con recepción
# No puedo respirar


## **13. Monitoreo y cierre**

El resumen muestra volumen, rutas, tiempos y errores. En producción, memoria, logs y deduplicación deben almacenarse de forma persistente y con controles adecuados para información sensible.


In [ ]:
from collections import Counter
def resumen_metricas():
    total=len(metricas)
    if not total: print("Aún no hay mensajes procesados."); return
    print("Mensajes:",total)
    print("Rutas:",dict(Counter(m["ruta"] for m in metricas)))
    print("Tiempo promedio:",round(sum(m["duracion_s"] for m in metricas)/total,1),"s")
    print("Errores:",len(errores_procesamiento))
resumen_metricas()


### **Dependencias y variables para producción**

Se conservan las mismas familias de librerías del ejemplo. El archivo `.env.example` documenta los secretos necesarios sin incluir valores.


In [ ]:
!pip freeze | grep -iE "^(fastapi|uvicorn|requests|llama[-_]stack|llama[-_]stack[-_]client|sentence[-_]transformers|ollama|transformers|torch)==" > requirements.txt
!cat requirements.txt


In [ ]:
variables = """VERIFY_TOKEN=
WHATSAPP_ACCESS_TOKEN=
PHONE_NUMBER_ID=
WABA_TOKEN_ID=
HF_TOKEN=
NGROK_AUTHTOKEN=
"""
with open(".env.example","w") as f: f.write(variables)
print(variables)


Al terminar, cierra el túnel para que la URL pública deje de estar disponible.


In [ ]:
ngrok.kill()
print("Proceso de ngrok cerrado.")
